# Phase 3A - Daily CMIP6 Archive Quality Control

This notebook checks the newly exported daily CMIP6 archive before bias correction and ETCCDI analysis. It validates file completeness, corrupt/empty files, expected daily time counts, variable names, dimensions, coordinate ranges, and basic value ranges.

Run this notebook before any daily QDM, ETCCDI, or trend analysis.

## Cell 1 - Imports and Paths

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl

warnings.filterwarnings('ignore')

ROOT = (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
DAILY_ROOT = ROOT / 'output' / 'cmip6_daily'
OUT_ROOT = ROOT / 'output' / 'daily_data_qc'
TABLE_DIR = OUT_ROOT / 'tables'
FIG_DIR = OUT_ROOT / 'figures'
LOG_DIR = OUT_ROOT / 'logs'

for d in [TABLE_DIR, FIG_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FULL_MODELS = ['CanESM5', 'GFDL-ESM4', 'INM-CM5-0', 'IPSL-CM6A-LR', 'MPI-ESM1-2-HR']
PRECIP_ONLY_MODELS = ['CESM2']
VARIABLES_BY_MODEL = {m: ['pr', 'tasmax', 'tasmin'] for m in FULL_MODELS}
for m in PRECIP_ONLY_MODELS:
    VARIABLES_BY_MODEL[m] = ['pr']

SCENARIOS = {
    'historical': (1985, 2014),
    'ssp245': (2015, 2100),
    'ssp585': (2015, 2100),
}

print(f'Daily root: {DAILY_ROOT}')
print(f'QC output: {OUT_ROOT}')

## Cell 2 - Build Expected File Table

In [ ]:
def expected_days(year):
    return len(pd.date_range(f'{year}-01-01', f'{year}-12-31', freq='D'))


def expected_file(model, scenario, variable, year):
    return DAILY_ROOT / scenario / model / variable / f'{model}_{variable}_{scenario}_{year}_daily.nc'


expected_rows = []
for scenario, (start, end) in SCENARIOS.items():
    for model, variables in VARIABLES_BY_MODEL.items():
        for variable in variables:
            for year in range(start, end + 1):
                path = expected_file(model, scenario, variable, year)
                expected_rows.append({
                    'scenario': scenario,
                    'model': model,
                    'variable': variable,
                    'year': year,
                    'expected_days': expected_days(year),
                    'path': str(path.relative_to(ROOT)),
                    'exists': path.exists(),
                    'size_bytes': path.stat().st_size if path.exists() else 0,
                })

expected_df = pd.DataFrame(expected_rows)
expected_df['too_small'] = expected_df['exists'] & (expected_df['size_bytes'] < 1_000_000)
expected_df.to_csv(TABLE_DIR / 'daily_expected_file_inventory.csv', index=False)

print('Expected files:', len(expected_df))
print('Existing files:', int(expected_df['exists'].sum()))
print('Missing files:', int((~expected_df['exists']).sum()))
print('Suspicious small files:', int(expected_df['too_small'].sum()))
expected_df.head()

## Cell 3 - Open and Validate Each Daily File

In [ ]:
def first_data_var(ds):
    return list(ds.data_vars)[0] if ds.data_vars else None


def value_bounds_ok(variable, vmin, vmax):
    if not np.isfinite(vmin) or not np.isfinite(vmax):
        return False
    if variable == 'pr':
        return (vmin >= -1e-4) and (vmax < 2000)
    if variable in ['tasmax', 'tasmin']:
        return (vmin > -80) and (vmax < 80)
    return True


qc_rows = []
bad_paths = []

for row in expected_df.itertuples(index=False):
    path = ROOT / row.path
    rec = row._asdict()
    rec.update({
        'open_ok': False,
        'data_var': None,
        'n_time': np.nan,
        'n_lat': np.nan,
        'n_lon': np.nan,
        'time_start': None,
        'time_end': None,
        'units': None,
        'min_value': np.nan,
        'max_value': np.nan,
        'mean_value': np.nan,
        'time_count_ok': False,
        'var_name_ok': False,
        'value_range_ok': False,
        'error': None,
    })

    if not row.exists:
        rec['error'] = 'missing_file'
        qc_rows.append(rec)
        bad_paths.append(row.path)
        continue

    try:
        with xr.open_dataset(path) as ds:
            var = row.variable if row.variable in ds.data_vars else first_data_var(ds)
            da = ds[var]
            rec['open_ok'] = True
            rec['data_var'] = var
            rec['var_name_ok'] = var == row.variable
            rec['n_time'] = int(ds.sizes.get('time', 0))
            rec['n_lat'] = int(ds.sizes.get('lat', ds.sizes.get('latitude', 0)))
            rec['n_lon'] = int(ds.sizes.get('lon', ds.sizes.get('longitude', 0)))
            rec['time_count_ok'] = rec['n_time'] == row.expected_days
            if 'time' in ds.coords and rec['n_time'] > 0:
                rec['time_start'] = str(pd.to_datetime(ds['time'].values[0]).date())
                rec['time_end'] = str(pd.to_datetime(ds['time'].values[-1]).date())
            rec['units'] = da.attrs.get('units')
            sample = da
            rec['min_value'] = float(sample.min(skipna=True).values)
            rec['max_value'] = float(sample.max(skipna=True).values)
            rec['mean_value'] = float(sample.mean(skipna=True).values)
            rec['value_range_ok'] = value_bounds_ok(row.variable, rec['min_value'], rec['max_value'])
    except Exception as exc:
        rec['error'] = f'{type(exc).__name__}: {exc}'

    if not (rec['open_ok'] and rec['time_count_ok'] and rec['var_name_ok'] and rec['value_range_ok']):
        bad_paths.append(row.path)
    qc_rows.append(rec)

qc_df = pd.DataFrame(qc_rows)
qc_df.to_csv(TABLE_DIR / 'daily_file_qc_full.csv', index=False)

bad_df = qc_df[~(qc_df['open_ok'] & qc_df['time_count_ok'] & qc_df['var_name_ok'] & qc_df['value_range_ok'])].copy()
bad_df.to_csv(TABLE_DIR / 'daily_file_qc_failed_or_suspicious.csv', index=False)
(LOG_DIR / 'files_to_redownload.txt').write_text('\n'.join(bad_df['path'].astype(str).tolist()), encoding='utf-8')

print('QC rows:', len(qc_df))
print('Failed/suspicious:', len(bad_df))
bad_df[['scenario', 'model', 'variable', 'year', 'size_bytes', 'open_ok', 'time_count_ok', 'value_range_ok', 'error', 'path']].head(30)

## Cell 4 - Completeness Tables

In [ ]:
qc_df['structurally_valid'] = qc_df['exists'] & (~qc_df['too_small']) & qc_df['open_ok'] & qc_df['time_count_ok'] & qc_df['var_name_ok']
qc_df['value_warning'] = qc_df['structurally_valid'] & (~qc_df['value_range_ok'])
qc_df['valid_file'] = qc_df['structurally_valid']

summary = (
    qc_df.groupby(['scenario', 'model', 'variable'], as_index=False)
    .agg(
        expected_files=('path', 'count'),
        existing_files=('exists', 'sum'),
        structurally_valid_files=('structurally_valid', 'sum'),
        value_warnings=('value_warning', 'sum'),
        suspicious_small=('too_small', 'sum'),
        first_year=('year', 'min'),
        last_year=('year', 'max'),
        total_size_gb=('size_bytes', lambda x: round(x.sum() / 1024**3, 3)),
    )
)
summary['missing_or_bad'] = summary['expected_files'] - summary['structurally_valid_files']
summary.to_csv(TABLE_DIR / 'daily_completeness_by_scenario_model_variable.csv', index=False)

scenario_summary = (
    qc_df.groupby(['scenario'], as_index=False)
    .agg(expected_files=('path', 'count'), structurally_valid_files=('structurally_valid', 'sum'), value_warnings=('value_warning', 'sum'), total_size_gb=('size_bytes', lambda x: round(x.sum() / 1024**3, 3)))
)
scenario_summary['missing_or_bad'] = scenario_summary['expected_files'] - scenario_summary['structurally_valid_files']
scenario_summary.to_csv(TABLE_DIR / 'daily_completeness_by_scenario.csv', index=False)

summary.sort_values(['missing_or_bad', 'scenario', 'model', 'variable'], ascending=[False, True, True, True]).head(30)

## Cell 5 - QC Figures

In [ ]:
mpl.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 140,
})

fig, ax = plt.subplots(figsize=(8, 4))
plot_df = scenario_summary.set_index('scenario')[['structurally_valid_files', 'missing_or_bad']]
plot_df.plot(kind='bar', stacked=True, color=['#2C7FB8', '#D95F0E'], ax=ax)
ax.set_ylabel('File count')
ax.set_xlabel('Scenario')
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / 'daily_qc_file_completeness_by_scenario.png', bbox_inches='tight')
plt.show()

heat = summary.pivot_table(index=['model', 'variable'], columns='scenario', values='missing_or_bad', aggfunc='sum').fillna(0)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(heat.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels([f'{m} | {v}' for m, v in heat.index])
ax.set_title('Missing or Failed Daily Files')
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, int(heat.values[i, j]), ha='center', va='center', color='black')
fig.colorbar(im, ax=ax, label='count')
fig.tight_layout()
fig.savefig(FIG_DIR / 'daily_qc_missing_bad_heatmap.png', bbox_inches='tight')
plt.show()

## Cell 6 - Quick Climate Sanity Plots

In [ ]:
valid_sample = qc_df[qc_df['valid_file']].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, variable in zip(axes, ['pr', 'tasmax', 'tasmin']):
    sub = valid_sample[valid_sample['variable'] == variable]
    if len(sub) == 0:
        ax.set_title(f'{variable}: no valid files')
        continue
    sub.boxplot(column='mean_value', by='scenario', ax=ax, grid=False)
    ax.set_title(variable)
    ax.set_xlabel('')
    ax.set_ylabel('file mean')
fig.suptitle('')
fig.tight_layout()
fig.savefig(FIG_DIR / 'daily_qc_file_mean_distributions.png', bbox_inches='tight')
plt.show()

## Cell 7 - Write Manuscript-Oriented QC Summary

In [ ]:
n_expected = len(qc_df)
n_valid = int(qc_df['valid_file'].sum())
n_bad = n_expected - n_valid
total_gb = qc_df['size_bytes'].sum() / 1024**3

struct_bad_df = qc_df[~qc_df['structurally_valid']].copy()
value_warn_df = qc_df[qc_df['value_warning']].copy()
struct_bad_df.to_csv(TABLE_DIR / 'daily_file_qc_structural_failures.csv', index=False)
value_warn_df.to_csv(TABLE_DIR / 'daily_file_qc_value_warnings.csv', index=False)
(LOG_DIR / 'files_to_redownload_structural_only.txt').write_text('\n'.join(struct_bad_df['path'].astype(str).tolist()), encoding='utf-8')

bad_lines = []
if len(struct_bad_df):
    for r in struct_bad_df[['scenario', 'model', 'variable', 'year', 'size_bytes', 'error', 'path']].itertuples(index=False):
        bad_lines.append(f'- {r.scenario} | {r.model} | {r.variable} | {r.year} | {r.size_bytes} bytes | {r.error or "structural validation failed"} | `{r.path}`')
else:
    bad_lines.append('- None')

summary_text = f'''# Phase 3A Daily CMIP6 Data QC Summary

## Purpose

This QC step validates the daily CMIP6 archive before daily QDM bias correction and ETCCDI extreme-index computation.

## Archive Checked

- Root: `output/cmip6_daily/`
- Full-variable models: {', '.join(FULL_MODELS)}
- Precipitation-only model: {', '.join(PRECIP_ONLY_MODELS)}
- Scenarios: historical, ssp245, ssp585
- Variables: pr, tasmax, tasmin where available

## QC Result

- Expected files: {n_expected}
- Structurally valid files: {int(qc_df['structurally_valid'].sum())}
- Missing or structurally failed files: {len(struct_bad_df)}
- Value-range warnings retained for analysis: {len(value_warn_df)}
- Total archive size checked: {total_gb:.2f} GB

## Files Needing Redownload or Repair

{chr(10).join(bad_lines)}

Value-range warnings are not automatically treated as corrupt files. They are retained for analysis but documented separately because daily precipitation maxima can be large in individual grid cells after unit conversion and clipping.

## Tables and Figures

- `tables/daily_expected_file_inventory.csv`
- `tables/daily_file_qc_full.csv`
- `tables/daily_file_qc_failed_or_suspicious.csv`
- `tables/daily_file_qc_structural_failures.csv`
- `tables/daily_file_qc_value_warnings.csv`
- `tables/daily_completeness_by_scenario_model_variable.csv`
- `tables/daily_completeness_by_scenario.csv`
- `logs/files_to_redownload.txt`
- `figures/daily_qc_file_completeness_by_scenario.png`
- `figures/daily_qc_missing_bad_heatmap.png`
- `figures/daily_qc_file_mean_distributions.png`

## Next Step

Proceed to daily QDM using structurally valid files. Any structurally failed future-year files can be skipped or redownloaded; value warnings should be reviewed but do not block analysis by default.
'''

(OUT_ROOT / 'PHASE_3A_DAILY_QC_SUMMARY.md').write_text(summary_text, encoding='utf-8')
print(summary_text)